In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch  # 导入 PyTorch 深度学习框架，提供张量运算和 GPU 支持

from transformers import (  # 从 transformers 库批量导入所需组件
    AutoModelForCausalLM,  # 自动加载因果语言模型（根据 config.json 自动选择架构）
    AutoTokenizer,  # 自动加载与模型匹配的分词器
    TrainingArguments,  # 训练超参数配置类（输出目录、batch_size、学习率等）
    Trainer,  # 高级训练器，封装了训练循环、梯度累积、日志记录等逻辑
    DataCollatorForLanguageModeling,  # CLM 专用数据整理器，自动处理 labels 生成和 padding
)

from peft import (  # 从 PEFT 库导入参数高效微调相关组件
    LoraConfig,  # LoRA 适配器的配置类（rank、alpha、target_modules 等）
    get_peft_model,  # 将 LoRA 适配器注入到基座模型中的工厂函数
    TaskType,  # 任务类型枚举，告诉 PEFT 当前是因果语言建模任务
    PeftModel,
)
from datasets import Dataset  # 导入 HuggingFace Dataset 类，用于构建和管理训练数据集

import pprint


In [2]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
lora_adapter = "./1/adapter"
lora_tokenizer = "./1/tokenizer"
output_dir = "./1/output_dir"

In [3]:
# ==================== 加载 Tokenizer ====================
# 从预训练仓库加载分词器
# 作用：将原始文本转换为模型可理解的 token id 序列
# 参数：
#   model_name: str - 模型仓库名称，用于下载 tokenizer_config.json / vocab 等文件
#   trust_remote_code: bool - 允许执行模型仓库中的自定义 Python 代码（Qwen 需要）
# 返回：PreTrainedTokenizerFast 实例
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
pprint.pprint(type(tokenizer))

<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>


In [4]:
# 加载基座模型
# 作用：加载 Qwen2.5-1.5B-Instruct 的全部权重作为 LoRA 微调的基座
# 参数：
#   model_name: str - 模型仓库名称
#   dtype: torch.dtype - 模型权重的数据类型，bfloat16 节省显存且保持精度
#   device_map: dict - 指定各层放置的设备
#   trust_remote_code: bool - 允许执行模型仓库中的自定义代码（Qwen 的 modeling 文件需要）
# 返回：Qwen2ForCausalLM 模型实例
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
pprint.pprint(type(model))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>


In [5]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
pprint.pprint(f"📍 使用设备: {device}")
pprint.pprint(model.device)

'📍 使用设备: cuda:0'
device(type='cuda', index=0)


#### 加载 lora

In [6]:
from peft import PeftModel

model = PeftModel.from_pretrained(model, lora_adapter)

In [7]:
def build_prompt(processor, utext: str) -> dict:
    messages = [{"role": "user", "content": utext}]
    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,          # 先不切 token，返回纯字符串
        add_generation_prompt=True,
    )
    inputs = processor(
        text=[text_prompt],   # 渲染后的 prompt 文本（列表，支持 batch）
        return_tensors="pt",  # 返回张量格式："pt"=PyTorch
    )
    return inputs

def move_to_device(inputs: dict, device: str) -> dict:
    new_inputs = {}

    for k, v in inputs.items():
        if isinstance(v, torch.Tensor):
            new_v = v.to(device)
        else:
            new_v = v
        new_inputs[k] = new_v

    return new_inputs

In [9]:
model.eval()
# prompt1 = "我家小狗叫什么"
prompt1 = "我家狗狗的名字是什么？"
inputs1 = build_prompt(tokenizer,prompt1)
inputs1 = move_to_device(inputs1,device)
model.eval()
with torch.no_grad():
    outputs1 = model.generate(
        **inputs1,
        num_return_sequences=1,
        do_sample=False,
        max_new_tokens=100, 
    )
pprint.pprint(tokenizer.decode(outputs1[0], skip_special_tokens=True))

('system\n'
 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.\n'
 'user\n'
 '我家狗狗的名字是什么？\n'
 'assistant\n'
 '它的名字叫naughty')
